# 🎤 Train Tiny Audio on LoquaciousSet

This notebook trains the Tiny Audio ASR model on **speechbrain/LoquaciousSet** - the same 25,000-hour dataset used to train the original model.

**Architecture:**
```
Audio (16kHz) → Whisper Encoder (frozen) → MLP Projector (trained) → SmolLM3-3B (frozen) → Text
```

**Citation:**
```bibtex
@software{kroman2025tinyaudio,
  author = {Kroman, Alex},
  title = {Tiny Audio: Train Your Own Speech Recognition Model in 24 Hours},
  year = {2025},
  url = {https://github.com/alexkroman/tiny-audio}
}
```

## Step 1: Install Dependencies

In [ ]:
# Uncomment and run if dependencies are not installed
# !pip install torch torchaudio transformers datasets accelerate evaluate jiwer truecase trl

# For faster HuggingFace downloads
# !pip install hf-transfer
# import os
# os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

## Step 2: Import Libraries & Setup

In [1]:
import os
import sys
from pathlib import Path

# Add src to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import torch
import truecase
from datasets import load_dataset, Audio
from transformers import Trainer, TrainingArguments
from trl.trainer.utils import DataCollatorForChatML

from src.asr_config import ASRConfig
from src.asr_modeling import ASRModel

# Enable TF32 for faster training
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

/home/jovyan/tiny-audio/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.8.0+cu128
CUDA: True
GPU: NVIDIA H100 80GB HBM3
Memory: 84.9 GB


## Step 3: Configuration (LoquaciousSet)

**LoquaciousSet** contains 25,000 hours of speech from:
- CommonVoice
- VoxPopuli  
- Libriheavy
- People's Speech
- YODAS

Subsets: `clean` (small), `medium` (recommended), `large` (full)

In [5]:
# ============== LOQUACIOUSSET CONFIGURATION ==============
DATASET_NAME = "speechbrain/LoquaciousSet"
DATASET_CONFIG = "small"   # Options: clean, medium, large
AUDIO_COLUMN = "wav"        # LoquaciousSet uses 'wav' for audio
TEXT_COLUMN = "text"        # LoquaciousSet uses 'text' for transcription

# ============== TRAINING CONFIGURATION ==============
SAMPLE_RATE = 16000
MAX_TRAIN_SAMPLES = 1000    # Start small! Set None for full dataset
MAX_EVAL_SAMPLES = 100
USE_STREAMING = False        # Required for large datasets like LoquaciousSet

# Training hyperparameters (optimized for A100 40GB)
BATCH_SIZE = 8              # A100: 4-6, H100: 6-8, reduce if OOM
LEARNING_RATE = 3e-4        # Optimal for projector training
MAX_STEPS = 500             # Quick test: 500, Full: 50000+
WARMUP_STEPS = 50           # ~10% of max_steps
EVAL_STEPS = 100
SAVE_STEPS = 100

# Output
OUTPUT_DIR = "./outputs/tiny-audio-loquacious"

# HuggingFace Hub (optional)
PUSH_TO_HUB = False
HUB_MODEL_ID = "your-username/tiny-audio-custom"

print("✅ Configuration set!")
print(f"   Dataset: {DATASET_NAME} ({DATASET_CONFIG})")
print(f"   Train samples: {MAX_TRAIN_SAMPLES or 'ALL'}")
print(f"   Max steps: {MAX_STEPS}")
print(f"   Batch size: {BATCH_SIZE}")

✅ Configuration set!
   Dataset: speechbrain/LoquaciousSet (small)
   Train samples: 1000
   Max steps: 500
   Batch size: 8


## Step 4: Load LoquaciousSet Dataset

In [6]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"  # Faster downloads

print(f"Loading {DATASET_NAME} ({DATASET_CONFIG})...")

# Load WITHOUT streaming - download the data
train_dataset = load_dataset(
    DATASET_NAME,
    name=DATASET_CONFIG,
    split="train",
    streaming=False,  # Changed from True
    trust_remote_code=True,
    num_proc=4,  # Parallel download
)

eval_dataset = load_dataset(
    DATASET_NAME,
    name=DATASET_CONFIG,
    split="dev",
    streaming=False,
    trust_remote_code=True,
    num_proc=4,
)

# Rename columns
train_dataset = train_dataset.rename_column(AUDIO_COLUMN, "audio")
eval_dataset = eval_dataset.rename_column(AUDIO_COLUMN, "audio")

# Cast audio to 16kHz
train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
eval_dataset = eval_dataset.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))

# Limit samples
if MAX_TRAIN_SAMPLES:
    train_dataset = train_dataset.select(range(min(MAX_TRAIN_SAMPLES, len(train_dataset))))
if MAX_EVAL_SAMPLES:
    eval_dataset = eval_dataset.select(range(min(MAX_EVAL_SAMPLES, len(eval_dataset))))

print("✅ Dataset loaded!")
print(f"   Train samples: {len(train_dataset)}")
print(f"   Eval samples: {len(eval_dataset)}")

Loading speechbrain/LoquaciousSet (small)...








































































































































































































































































































































































































































Generating train split: 107303 examples [00:11, 9230.38 examples/s] 
Generating dev split: 7759 examples [00:01, 6361.38 examples/s] 
Generating test split: 8087 examples [00:12, 661.11 examples/s] 


✅ Dataset loaded!
   Train samples: 1000
   Eval samples: 100


### Preview a Sample

In [7]:
# Preview first sample
sample = next(iter(train_dataset))
print("Transcription:", sample["text"])
print("Audio shape:", sample["audio"]["array"].shape)
print("Sample rate:", sample["audio"]["sampling_rate"])

Transcription: AND WHAT ABOUT INTEROPERABILITY IN THE RAIL SECTOR ARE NATIONAL BARRIERS PREVENTING PROGRESS IN THIS AREA AS WELL OR IS THERE AN UNWILLINGNESS ON THE PART OF THE RAIL INDUSTRY TO EMBRACE THE CONCEPT OF INTEROPERABILITY
Audio shape: (273920,)
Sample rate: 16000


## Step 5: Initialize Model

Creates the ASR model:
- **Whisper Large V3 Turbo** (frozen) - audio encoder
- **MLP Projector** (~12M params, trainable)
- **SmolLM3-3B** (frozen) - text decoder

In [8]:
config = ASRConfig(
    audio_model_id="openai/whisper-large-v3-turbo",
    text_model_id="HuggingFaceTB/SmolLM3-3B",
    projector_type="mlp",
    system_prompt="/no_think /system_override",
    attn_implementation="sdpa",
)

print("Creating model... (downloading ~15GB of model weights)")
model = ASRModel(config)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✅ Model created!")
print(f"   Total: {total_params / 1e9:.2f}B parameters")
print(f"   Trainable: {trainable_params / 1e6:.1f}M parameters ({100*trainable_params/total_params:.2f}%)")

Creating model... (downloading ~15GB of model weights)


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 107.92it/s]



✅ Model created!
   Total: 3.72B parameters
   Trainable: 11.7M parameters (0.32%)


## Step 6: Create Data Collator

In [9]:
TRANSCRIBE_PREFIX = "Transcribe: "

class DataCollator:
    def __init__(self, tokenizer, feature_extractor, sample_rate, system_prompt=None):
        self.tokenizer = tokenizer
        self.feature_extractor = feature_extractor
        self.sample_rate = sample_rate
        self.system_prompt = system_prompt
        self.text_collator = DataCollatorForChatML(tokenizer=tokenizer, max_length=2048)
    
    def __call__(self, features):
        audio_arrays, valid_features = [], []
        
        for f in features:
            try:
                audio = f["audio"]["array"]
                if hasattr(audio, "numpy"):
                    audio = audio.numpy()
                audio = audio.squeeze()
                if audio.ndim > 1:
                    audio = audio.mean(axis=0)
                audio_arrays.append(audio)
                valid_features.append(f)
            except:
                continue
            finally:
                f["audio"] = None
        
        if not audio_arrays:
            raise ValueError("No valid audio in batch")
        
        audio_out = self.feature_extractor(
            audio_arrays, sampling_rate=self.sample_rate,
            padding="max_length", return_tensors="pt"
        )
        
        mel_len = audio_out.input_features.shape[-1]
        num_audio_tokens = mel_len // 4
        audio_placeholder = "<audio>" * num_audio_tokens
        user_content = TRANSCRIBE_PREFIX + audio_placeholder
        
        text_features = []
        for f in valid_features:
            text = truecase.get_true_case((f.get("text") or "").strip())
            messages = []
            if self.system_prompt:
                messages.append({"role": "system", "content": self.system_prompt})
            messages.append({"role": "user", "content": user_content})
            messages.append({"role": "assistant", "content": text})
            text_features.append({"messages": messages})
        
        batch = self.text_collator(text_features)
        batch["input_features"] = audio_out.input_features
        return batch

data_collator = DataCollator(
    tokenizer=model.tokenizer,
    feature_extractor=model.feature_extractor,
    sample_rate=SAMPLE_RATE,
    system_prompt=config.system_prompt,
)
print("✅ Data collator created!")

✅ Data collator created!


## Step 7: Configure Training

In [10]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    
    # Training
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    max_steps=MAX_STEPS,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.1,
    
    # Precision
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    gradient_accumulation_steps=4,
    
    # Evaluation & Saving
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    
    # Logging
    logging_steps=10,
    report_to="none",  # Change to "wandb" for W&B logging
    
    # Hub
    push_to_hub=PUSH_TO_HUB,
    hub_model_id=HUB_MODEL_ID if PUSH_TO_HUB else None,
    
    # Other
    remove_unused_columns=False,
    dataloader_num_workers=2,
)
print("✅ Training arguments configured!")

✅ Training arguments configured!


In [11]:
import nltk 
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/jovyan/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [11]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /home/jovyan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Step 8: Train! 🚀

In [12]:
model.config.use_cache = False

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("Starting training...")
print("="*50)
trainer.train()
print("="*50)
print("✅ Training complete!")

Starting training...


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
100,3.078600,3.165173
200,2.543400,3.163235
300,0.500300,0.812960
400,0.139600,0.728267
500,0.101200,0.763569


There were missing keys in the checkpoint model loaded: ['audio_tower.conv1.weight', 'audio_tower.conv1.bias', 'audio_tower.conv2.weight', 'audio_tower.conv2.bias', 'audio_tower.embed_positions.weight', 'audio_tower.layers.0.self_attn.k_proj.weight', 'audio_tower.layers.0.self_attn.v_proj.weight', 'audio_tower.layers.0.self_attn.v_proj.bias', 'audio_tower.layers.0.self_attn.q_proj.weight', 'audio_tower.layers.0.self_attn.q_proj.bias', 'audio_tower.layers.0.self_attn.out_proj.weight', 'audio_tower.layers.0.self_attn.out_proj.bias', 'audio_tower.layers.0.self_attn_layer_norm.weight', 'audio_tower.layers.0.self_attn_layer_norm.bias', 'audio_tower.layers.0.fc1.weight', 'audio_tower.layers.0.fc1.bias', 'audio_tower.layers.0.fc2.weight', 'audio_tower.layers.0.fc2.bias', 'audio_tower.layers.0.final_layer_norm.weight', 'audio_tower.layers.0.final_layer_norm.bias', 'audio_tower.layers.1.self_attn.k_proj.weight', 'audio_tower.layers.1.self_attn.v_proj.weight', 'audio_tower.layers.1.self_attn.v_p

✅ Training complete!


## Step 9: Save Model

In [13]:
trainer.save_model()
print(f"✅ Model saved to {OUTPUT_DIR}")

# if PUSH_TO_HUB:
#     trainer.push_to_hub(commit_message="Training complete")
#     print(f"✅ Pushed to HuggingFace: {HUB_MODEL_ID}")

✅ Model saved to ./outputs/tiny-audio-loquacious


## Step 10: Test Your Model! 🎉

In [16]:
# Test using the model already in memory
model.eval()
model.config.use_cache = True

# Get a test sample
test_sample = eval_dataset[0]
audio_array = test_sample["audio"]["array"]
ground_truth = test_sample["text"]

# Prepare input - match model dtype!
inputs = model.feature_extractor(
    audio_array, 
    sampling_rate=16000, 
    return_tensors="pt"
).input_features.to(model.device).to(model.dtype)  # <-- Added .to(model.dtype)

# Generate transcription
with torch.no_grad():
    output = model.generate(input_features=inputs, max_new_tokens=256)

# Decode
transcription = model.tokenizer.decode(output[0], skip_special_tokens=True)

print("="*50)
print("Ground truth:", ground_truth)
print("Prediction:  ", transcription)
print("="*50)

Ground truth: THESE ARE REFORMS THAT WILL DISCIPLINE AND CONSTRAIN THE EXERCISE OF POWER BY THE GOVERNMENT AND ANY OTHER ECONOMIC OR POLITICAL ACTOR FOR GENERATIONS TO COME
Prediction:   These are reforms that will discipline and constrain the exercise of power by the government and any other economic or political actor for generations to come


---

## What's Next?

1. **Full training**: Set `MAX_TRAIN_SAMPLES = None` and `MAX_STEPS = 50000+`
2. **Different projector**: Change `projector_type` to `moe`, `swiglu`, or `residual`
3. **Evaluate WER**: `poetry run eval ./outputs/tiny-audio-loquacious`

## Citation

```bibtex
@misc{loquaciousset2024,
  author = {{SpeechBrain Team}},
  title = {LoquaciousSet: A Large-Scale Speech Recognition Dataset},
  url = {https://huggingface.co/datasets/speechbrain/LoquaciousSet}
}
```